# Notebook 14 — Online Drift Detection

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 13 decomposed mixed-regime windows using prototype mixtures.

Notebook 14 detects when the stream stops matching those prototypes:

- reconstruction residual drift
- mixture entropy drift
- policy instability
- unknown-regime candidates
- rolling drift alarms

Constraint view:
> when reconstruction residuals rise, existing execution prototypes no longer explain the stream.

## Goals

1. Load Notebook 13 mixed-regime decomposition results when available.
2. Compute rolling baseline statistics.
3. Build drift scores from:
   - reconstruction residual
   - mixture entropy
   - policy switching
   - dominant-regime instability
4. Mark drift alarms.
5. Identify unknown-regime candidate windows.
6. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 13 mixed-regime decomposition

If missing, this notebook creates a fallback table with synthetic drift injected.

In [ ]:
mix_path = RESULTS_DIR / "notebook13_mixed_regime_decomposition.csv"

if mix_path.exists():
    df = pd.read_csv(mix_path)
    print("Loaded:", mix_path)
else:
    print("Notebook 13 output not found; creating fallback drift table.")
    rng = np.random.default_rng(42)
    n = 180
    regimes = ["low_entropy_repeating", "sequential_ids", "uniform_32bit", "zipfian_smallints", "clustered_ranges"]
    policies = ["coherent_local", "hybrid", "simd", "hybrid", "guarded_fallback"]
    rows = []
    for i in range(n):
        phase = i % 40
        regime = regimes[(i // 12) % len(regimes)]
        policy = policies[regimes.index(regime)]
        residual = abs(rng.normal(0.035, 0.015))
        entropy = abs(rng.normal(1.05, 0.28))

        # Inject unknown/drift interval.
        if 105 <= i <= 135:
            residual += 0.08 + 0.04 * rng.random()
            entropy += 0.70 + 0.30 * rng.random()
            if i % 3 == 0:
                policy = "hybrid"
            elif i % 3 == 1:
                policy = "simd"
            else:
                policy = "guarded_fallback"

        rows.append({
            "window_id": i,
            "dominant_truth_regime": regime,
            "estimated_dominant_regime": regime if rng.random() > 0.05 else rng.choice(regimes),
            "mixture_policy": policy,
            "mixture_entropy": entropy,
            "reconstruction_residual": residual,
        })
    df = pd.DataFrame(rows)

df.head()

## Normalize schema and compute rolling features

In [ ]:
work = df.copy().sort_values("window_id").reset_index(drop=True)

required_defaults = {
    "mixture_entropy": 0.0,
    "reconstruction_residual": 0.0,
}
for col, default in required_defaults.items():
    if col not in work.columns:
        work[col] = default
    work[col] = pd.to_numeric(work[col], errors="coerce").fillna(default)

if "mixture_policy" not in work.columns:
    work["mixture_policy"] = "unknown"
if "estimated_dominant_regime" not in work.columns:
    work["estimated_dominant_regime"] = "unknown"

window = 15
eps = 1e-9

# Rolling statistics use shifted windows so each score is compared to prior behavior.
work["residual_roll_mean"] = work["reconstruction_residual"].shift(1).rolling(window, min_periods=5).mean()
work["residual_roll_std"] = work["reconstruction_residual"].shift(1).rolling(window, min_periods=5).std().fillna(0)
work["entropy_roll_mean"] = work["mixture_entropy"].shift(1).rolling(window, min_periods=5).mean()
work["entropy_roll_std"] = work["mixture_entropy"].shift(1).rolling(window, min_periods=5).std().fillna(0)

# Fill early baselines.
work["residual_roll_mean"] = work["residual_roll_mean"].fillna(work["reconstruction_residual"].expanding().mean())
work["entropy_roll_mean"] = work["entropy_roll_mean"].fillna(work["mixture_entropy"].expanding().mean())
work["residual_roll_std"] = work["residual_roll_std"].replace(0, np.nan).fillna(work["reconstruction_residual"].std() + eps)
work["entropy_roll_std"] = work["entropy_roll_std"].replace(0, np.nan).fillna(work["mixture_entropy"].std() + eps)

work["residual_z"] = (work["reconstruction_residual"] - work["residual_roll_mean"]) / (work["residual_roll_std"] + eps)
work["entropy_z"] = (work["mixture_entropy"] - work["entropy_roll_mean"]) / (work["entropy_roll_std"] + eps)

# Instability signals.
work["policy_changed"] = work["mixture_policy"].ne(work["mixture_policy"].shift(1)).fillna(False)
work["dominant_regime_changed"] = work["estimated_dominant_regime"].ne(work["estimated_dominant_regime"].shift(1)).fillna(False)

work["policy_switch_rate"] = work["policy_changed"].rolling(window, min_periods=1).mean()
work["dominant_switch_rate"] = work["dominant_regime_changed"].rolling(window, min_periods=1).mean()

work[[
    "window_id", "mixture_entropy", "reconstruction_residual",
    "residual_z", "entropy_z", "policy_switch_rate"
]].head()

## Combined drift score

The drift score combines:

- positive residual z-score
- positive entropy z-score
- policy-switch rate
- dominant-regime switch rate

It is not a proof of drift; it is an alarm score for windows that deserve inspection.

In [ ]:
def positive_clip(s, cap=4.0):
    return np.clip(pd.Series(s).astype(float), 0, cap) / cap

work["residual_drift_component"] = positive_clip(work["residual_z"])
work["entropy_drift_component"] = positive_clip(work["entropy_z"])

work["drift_score"] = (
    0.40 * work["residual_drift_component"] +
    0.25 * work["entropy_drift_component"] +
    0.20 * work["policy_switch_rate"] +
    0.15 * work["dominant_switch_rate"]
).clip(0, 1)

drift_threshold = 0.55
warning_threshold = 0.35

work["drift_warning"] = work["drift_score"] >= warning_threshold
work["drift_alarm"] = work["drift_score"] >= drift_threshold

# Unknown-regime candidate: high residual and high entropy together.
work["unknown_regime_candidate"] = (
    (work["residual_z"] > 2.0) &
    (work["entropy_z"] > 1.0)
) | (
    (work["drift_alarm"]) &
    (work["reconstruction_residual"] > work["reconstruction_residual"].quantile(0.85))
)

work[[
    "window_id", "drift_score", "drift_warning", "drift_alarm", "unknown_regime_candidate"
]].head()

## Segment drift intervals

Consecutive alarm windows are grouped into drift episodes.

In [ ]:
episode_ids = []
current = -1
in_episode = False

for alarm in work["drift_alarm"]:
    if alarm and not in_episode:
        current += 1
        in_episode = True
    elif not alarm:
        in_episode = False
    episode_ids.append(current if alarm else np.nan)

work["drift_episode_id"] = episode_ids

episodes = (
    work.dropna(subset=["drift_episode_id"])
    .groupby("drift_episode_id", as_index=False)
    .agg(
        start_window=("window_id", "min"),
        end_window=("window_id", "max"),
        duration=("window_id", "size"),
        max_drift_score=("drift_score", "max"),
        mean_residual=("reconstruction_residual", "mean"),
        mean_entropy=("mixture_entropy", "mean"),
        unknown_candidates=("unknown_regime_candidate", "sum"),
    )
)

episodes

## Export drift-detection tables

In [ ]:
csv_path = RESULTS_DIR / "notebook14_online_drift_detection.csv"
json_path = RESULTS_DIR / "notebook14_online_drift_detection.json"
episodes_csv_path = RESULTS_DIR / "notebook14_drift_episodes.csv"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)
episodes.to_csv(episodes_csv_path, index=False)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", episodes_csv_path)

## Figure 1 — Drift score timeline

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook14_drift_score_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["drift_score"], label="drift score")
plt.axhline(warning_threshold, linestyle="--", label="warning threshold")
plt.axhline(drift_threshold, linestyle="--", label="alarm threshold")
alarms = work[work["drift_alarm"]]
plt.scatter(alarms["window_id"], alarms["drift_score"], s=35, label="alarms")
plt.xlabel("Window")
plt.ylabel("Drift score")
plt.title("Online Drift Detection: Drift Score Timeline")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Residual and entropy z-scores

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook14_residual_entropy_zscores.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["residual_z"], label="residual z-score")
plt.plot(work["window_id"], work["entropy_z"], label="entropy z-score")
plt.axhline(2.0, linestyle="--", label="z = 2")
plt.xlabel("Window")
plt.ylabel("Z-score")
plt.title("Online Drift Detection: Residual and Mixture-Entropy Z-Scores")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Reconstruction residual with alarms

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook14_reconstruction_residual_alarms.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["reconstruction_residual"], label="reconstruction residual")
alarm_rows = work[work["drift_alarm"]]
plt.scatter(alarm_rows["window_id"], alarm_rows["reconstruction_residual"], s=35, label="drift alarms")
unknown = work[work["unknown_regime_candidate"]]
plt.scatter(unknown["window_id"], unknown["reconstruction_residual"], s=60, marker="x", label="unknown candidates")
plt.xlabel("Window")
plt.ylabel("NNLS reconstruction residual")
plt.title("Prototype Mismatch: Reconstruction Residual Alarms")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Policy and dominant-regime instability

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook14_policy_regime_instability.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["policy_switch_rate"], label="policy switch rate")
plt.plot(work["window_id"], work["dominant_switch_rate"], label="dominant-regime switch rate")
plt.xlabel("Window")
plt.ylabel("Rolling switch rate")
plt.title("Online Drift Detection: Policy and Regime Instability")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Drift alarm timeline

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook14_drift_alarm_timeline.png"

plt.figure(figsize=(12, 3))
plt.step(work["window_id"], work["drift_warning"].astype(int), where="mid", label="warning")
plt.step(work["window_id"], work["drift_alarm"].astype(int) + 1.2, where="mid", label="alarm")
plt.step(work["window_id"], work["unknown_regime_candidate"].astype(int) + 2.4, where="mid", label="unknown candidate")
plt.yticks([0, 1.2, 2.4], ["warning", "alarm", "unknown"])
plt.xlabel("Window")
plt.title("Online Drift Detection: Alarm Timeline")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Drift episodes summary

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook14_drift_episodes_summary.png"

if len(episodes):
    plt.figure(figsize=(9, 5))
    plt.bar(episodes["drift_episode_id"].astype(str), episodes["max_drift_score"])
    plt.xlabel("Drift episode")
    plt.ylabel("Max drift score")
    plt.title("Drift Episodes: Maximum Drift Score")
    plt.tight_layout()
    plt.savefig(fig_path_6, dpi=160)
    plt.show()
else:
    plt.figure(figsize=(7, 3))
    plt.text(0.5, 0.5, "No drift episodes detected", ha="center", va="center")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(fig_path_6, dpi=160)
    plt.show()

print("Saved:", fig_path_6)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_14_online_drift_detection.md"

summary = {
    "windows": int(len(work)),
    "warning_windows": int(work["drift_warning"].sum()),
    "alarm_windows": int(work["drift_alarm"].sum()),
    "unknown_candidate_windows": int(work["unknown_regime_candidate"].sum()),
    "drift_episodes": int(len(episodes)),
    "max_drift_score": float(work["drift_score"].max()),
    "mean_drift_score": float(work["drift_score"].mean()),
    "mean_reconstruction_residual": float(work["reconstruction_residual"].mean()),
    "mean_mixture_entropy": float(work["mixture_entropy"].mean()),
}

lines = [
    "# Report 14 — Online Drift Detection",
    "",
    "This report detects windows where existing mixed-regime prototypes no longer explain the stream.",
    "",
    "Constraint view:",
    "> when reconstruction residuals rise, existing execution prototypes no longer explain the stream.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Episodes CSV: `{episodes_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Drift episodes",
    "",
    episodes.to_markdown(index=False) if len(episodes) else "No drift episodes detected.",
    "",
    "## Interpretation",
    "",
    "- Reconstruction residuals measure prototype mismatch.",
    "- Mixture entropy measures ambiguity in the regime decomposition.",
    "- Policy and regime switch rates identify unstable runtime behavior.",
    "- Drift alarms mark windows where adaptation should slow down, inspect, or propose new prototypes.",
    "",
    "## Next step",
    "",
    "Notebook 15 can perform prototype update and recovery: learn a new prototype from drift windows and test whether residuals decrease.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook14_online_drift_detection_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook14_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_14_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))